In [128]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re
from scipy.signal import welch
from scipy.integrate import trapezoid
import time
import scipy.io as sio


# Step 1 

## Configuration

In [129]:
DATA_ROOT    = Path('../Data')
FILTERED_DIR = DATA_ROOT / 'filtered_data'
SCALES_PATH  = DATA_ROOT / 'scales.xls'

In [130]:
FS               = 128
N_CHANNELS       = 32
N_SAMPLES        = 3200       # 25 * 128
STRESS_THRESHOLD = 5
DATA_KEY    = "Clean_data"  # confirmed key in the .mat files
WIN_SEC     = 4.0
OVERLAP     = 0.5
WIN_SAMP    = int(WIN_SEC * FS)                 # 512
HOP_SAMP    = int(WIN_SAMP * (1 - OVERLAP))     # 256

In [131]:
TASK_TO_SCALE_COL = {
    "Arithmetic":  "Maths",
    "Mirror_image":"Symmetry",
    "Stroop":      "Stroop",
}

## Parse scales.xls into a tidy label table

In [132]:
scales = pd.read_excel(SCALES_PATH, header=[0, 1])
print("scales shape:", scales.shape)
print("Columns (MultiIndex):")
for c in scales.columns:
    print("  ", c)
scales.head()

scales shape: (40, 10)
Columns (MultiIndex):
   ('Subject No.', 'Unnamed: 0_level_1')
   ('Trial_1', 'Maths')
   ('Trial_1', 'Symmetry')
   ('Trial_1', 'Stroop')
   ('Trial_2', 'Maths')
   ('Trial_2', 'Symmetry')
   ('Trial_2', 'Stroop')
   ('Trial_3', 'Maths')
   ('Trial_3', 'Symmetry')
   ('Trial_3', 'Stroop')


Subject No. Trial_1                 Trial_2                 Trial_3  \
  Unnamed: 0_level_1   Maths Symmetry Stroop   Maths Symmetry Stroop   Maths   
0                  1       6        3      3       7        5      2       4   
1                  2       3        4      5       3        4      4       7   
2                  3       5        3      4       3        5      5       8   
3                  4       5        3      4       3        5      2       7   
4                  5       6        6      6       5        3      2       5   

                   
  Symmetry Stroop  
0        7      4  
1        5      3  
2        7      5  
3        5      5  
4        7      3

In [133]:
id_col_name = scales.columns[0]     
long_rows = []

for _, row in scales.iterrows():
    subject = int(row[id_col_name])
    for (trial_label, task_label), value in row.items():
        if (trial_label, task_label) == id_col_name:
            continue

        # Map the spreadsheet task label back to our MAT task names.
        mat_task = None
        for mt, scol in TASK_TO_SCALE_COL.items():
            if str(task_label).lower() == scol.lower():
                mat_task = mt
                break
        if mat_task is None:
            continue  # skip anything that isn't Maths/Symmetry/Stroop

        # Extract trial number from the second header level (e.g. "Trial 1" -> 1).
        digits = "".join(ch for ch in str(trial_label) if ch.isdigit())
        if digits == "":
            continue
        trial = int(digits)

        rating = float(value)
        long_rows.append({
            "subject": subject,
            "task": mat_task,
            "trial": trial,
            "rating": rating,
        })


labels = pd.DataFrame(long_rows)
labels["label"] = (labels["rating"] >= STRESS_THRESHOLD).astype(int)
labels

,subject,task,trial,rating,label
0,1,Arithmetic,1,6.0,1
1,1,Mirror_image,1,3.0,0
2,1,Stroop,1,3.0,0
3,1,Arithmetic,2,7.0,1
4,1,Mirror_image,2,5.0,1
...,...,...,...,...,...
355,40,Mirror_image,2,6.0,1
356,40,Stroop,2,5.0,1
357,40,Arithmetic,3,4.0,0
358,40,Mirror_image,3,4.0,0


In [134]:
pattern = re.compile(r"(?P<task>[A-Za-z_]+)_sub_(?P<subject>\d+)_trial(?P<trial>\d+)\.mat$")

records = []
search_dir = FILTERED_DIR if FILTERED_DIR.exists() else DATA_ROOT  # fall back to demo file
for f in sorted(search_dir.glob("*.mat")):
    m = pattern.match(f.name)
    if not m:
        continue
    task = m.group("task").rstrip("_")
    # normalize possible variants like "Mirror_image"
    records.append({
        "path": str(f),
        "task": task,
        "subject": int(m.group("subject")),
        "trial": int(m.group("trial")),
    })

manifest = pd.DataFrame(records)
print("Files found:", len(manifest))

# Join labels (only Arithmetic/Mirror_image/Stroop have ratings; Relax will be NaN).
manifest = manifest.merge(labels[["subject", "task", "trial", "rating", "label"]],
                          on=["subject", "task", "trial"], how="left")

print("With label:", manifest["label"].notna().sum(), "/ Relax or unmatched:", manifest["label"].isna().sum())

Files found: 480
With label: 360 / Relax or unmatched: 120


In [135]:
manifest['task'].value_counts()

task
Arithmetic      120
Mirror_image    120
Relax           120
Stroop          120
Name: count, dtype: int64

In [136]:
manifest.isna().sum()

path         0
task         0
subject      0
trial        0
rating     120
label      120
dtype: int64

In [137]:
manifest.dropna(inplace=True)

In [138]:
manifest.isna().sum()

path       0
task       0
subject    0
trial      0
rating     0
label      0
dtype: int64

In [139]:
manifest.info()

<class 'pandas.DataFrame'>
Index: 360 entries, 0 to 479
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   path     360 non-null    str    
 1   task     360 non-null    str    
 2   subject  360 non-null    int64  
 3   trial    360 non-null    int64  
 4   rating   360 non-null    float64
 5   label    360 non-null    float64
dtypes: float64(2), int64(2), str(2)
memory usage: 19.7 KB


In [140]:
manifest.head()

,path,task,subject,trial,rating,label
0,../Data/filtered_data/Arithmetic_sub_10_trial1...,Arithmetic,10,1,3.0,0.0
1,../Data/filtered_data/Arithmetic_sub_10_trial2...,Arithmetic,10,2,4.0,0.0
2,../Data/filtered_data/Arithmetic_sub_10_trial3...,Arithmetic,10,3,6.0,1.0
3,../Data/filtered_data/Arithmetic_sub_11_trial1...,Arithmetic,11,1,7.0,1.0
4,../Data/filtered_data/Arithmetic_sub_11_trial2...,Arithmetic,11,2,6.0,1.0


# Step 2 - Feature Extraction

In [141]:
BANDS = {                    # same 5 bands as the no-window baseline
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 45),
}
EPS = 1e-12

def band_power(psd, freqs, fmin, fmax):
    idx = np.logical_and(freqs >= fmin, freqs <= fmax)
    return trapezoid(psd[idx], freqs[idx]) if idx.any() else 0.0

def hjorth_params(x):
    """Return (mobility, complexity) of a 1-D signal."""
    dx  = np.diff(x)
    ddx = np.diff(dx)
    var_x, var_dx, var_ddx = np.var(x), np.var(dx), np.var(ddx)
    mobility   = np.sqrt(var_dx / (var_x + EPS))
    complexity = np.sqrt(var_ddx / (var_dx + EPS)) / (mobility + EPS)
    return mobility, complexity

def spectral_entropy(psd):
    """Shannon entropy of the normalized PSD, normalized to [0, 1]."""
    p = psd / (psd.sum() + EPS)
    return -np.sum(p * np.log2(p + EPS)) / np.log2(len(p))

def extract_features(data_2d, fs=FS):
    """
    data_2d: (n_channels, n_samples)
    Per channel: 5 log abs power + 5 relative power + 2 ratios
                 + 2 Hjorth + 1 spectral entropy = 14 features.
    """
    feats = []
    for ch in range(data_2d.shape[0]):
        x = data_2d[ch]
        freqs, psd = welch(x, fs=fs, nperseg=fs * 2)

        # absolute band powers
        bp = {b: band_power(psd, freqs, lo, hi) for b, (lo, hi) in BANDS.items()}
        total = sum(bp.values()) + EPS

        feats += [np.log10(bp[b] + EPS) for b in BANDS]        # 5 log abs power
        feats += [bp[b] / total for b in BANDS]                # 5 relative power
        feats += [bp["beta"]  / (bp["alpha"] + EPS),           # arousal index
                  bp["theta"] / (bp["alpha"] + EPS)]           # workload index
        feats += list(hjorth_params(x))                        # 2 Hjorth
        feats.append(spectral_entropy(psd))                    # 1 entropy
    return np.array(feats, dtype=np.float32)

def feature_names(channel_names):
    per_ch = ([f"logpow_{b}" for b in BANDS] +
              [f"relpow_{b}" for b in BANDS] +
              ["ratio_beta_alpha", "ratio_theta_alpha",
               "hjorth_mobility", "hjorth_complexity", "spec_entropy"])
    return [f"{ch}_{f}" for ch in channel_names for f in per_ch]


from scipy.stats import skew, kurtosis


# ---- Windowing --------------------------------------------------------------
def sliding_windows(sig, win, hop):
    """sig: (n_channels, n_samples) -> yields (n_channels, win) slices."""
    n_samples = sig.shape[1]
    for start in range(0, n_samples - win + 1, hop):
        yield sig[:, start:start + win]

BANDS = {
    "delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 13),
    "beta":  (13, 30), "gamma": (30, 45),
}

# Symmetric electrode pairs for hemispheric asymmetry (SAM-40 uses 10-20 EmotivEPOC+ layout).
# Adjust indices to your actual channel order.
ASYM_PAIRS = [(0, 1), (2, 3), (4, 5), (6, 7)]  # (left_idx, right_idx)

def _hjorth(x):
    # x: (n_channels, win)
    d1 = np.diff(x, axis=1)
    d2 = np.diff(d1, axis=1)
    var0 = np.var(x,  axis=1) + 1e-12
    var1 = np.var(d1, axis=1) + 1e-12
    var2 = np.var(d2, axis=1) + 1e-12
    activity   = var0
    mobility   = np.sqrt(var1 / var0)
    complexity = np.sqrt(var2 / var1) / mobility
    return activity, mobility, complexity

def extract_features(window, fs=FS):
    """window: (n_channels, win) -> rich flat feature vector."""
    nch = window.shape[0]
    freqs, psd = welch(window, fs=fs, nperseg=min(256, window.shape[1]), axis=1)

    # --- absolute band power (32 x 5 = 160) ---
    bp = {}
    for name, (lo, hi) in BANDS.items():
        idx = (freqs >= lo) & (freqs <= hi)
        bp[name] = trapezoid(psd[:, idx], freqs[idx], axis=1)   # (nch,)
    abs_bp = np.concatenate([bp[b] for b in BANDS])             # 160

    # --- relative band power (each band / total) (32 x 5 = 160) ---
    total = sum(bp.values()) + 1e-12
    rel_bp = np.concatenate([bp[b] / total for b in BANDS])     # 160

    # --- band-power ratios: theta/beta, alpha/beta, theta/alpha (32 x 3 = 96) ---
    ratios = np.concatenate([
        bp["theta"] / (bp["beta"]  + 1e-12),
        bp["alpha"] / (bp["beta"]  + 1e-12),
        bp["theta"] / (bp["alpha"] + 1e-12),
    ])                                                          # 96

    # --- Hjorth parameters (32 x 3 = 96) ---
    act, mob, comp = _hjorth(window)
    hjorth = np.concatenate([act, mob, comp])                   # 96

    # --- spectral entropy per channel (32) ---
    psd_norm = psd / (psd.sum(axis=1, keepdims=True) + 1e-12)
    spec_ent = -np.sum(psd_norm * np.log2(psd_norm + 1e-12), axis=1)  # 32

    # --- time-domain statistics (32 x 4 = 128) ---
    stats = np.concatenate([
        np.mean(window, axis=1),
        np.std(window,  axis=1),
        skew(window, axis=1),
        kurtosis(window, axis=1),
    ])                                                          # 128

    # --- hemispheric alpha asymmetry: ln(R) - ln(L) (len = len(ASYM_PAIRS)) ---
    asym = np.array([
        np.log(bp["alpha"][r] + 1e-12) - np.log(bp["alpha"][l] + 1e-12)
        for (l, r) in ASYM_PAIRS
    ])                                                          # 4

    return np.concatenate([abs_bp, rel_bp, ratios, hjorth,
                           spec_ent, stats, asym])

### Building full feature 

In [142]:
manifest.columns.tolist()

['path', 'task', 'subject', 'trial', 'rating', 'label']

In [143]:
from scipy.io import loadmat

# ---- Build windowed dataset -------------------------------------------------
# meta_df must map each recording to its file path, subject id, and label.
# Reusing features_meta.csv from the baseline (columns: filepath, subject, label).
# meta_df = pd.read_csv("features_meta.csv")
meta_df = manifest[['path', 'subject', 'label']]

X_win, y_win, groups_win, rec_ids = [], [], [], []

for rec_idx, row in meta_df.iterrows():
    sig = loadmat(row["path"])[DATA_KEY].astype(np.float64)  # (32, 3200)
    for w in sliding_windows(sig, WIN_SAMP, HOP_SAMP):
        X_win.append(extract_features(w))
        y_win.append(int(row["label"]))
        groups_win.append(int(row["subject"]))   # <-- subject ID, NOT window/recording index
        rec_ids.append(rec_idx)                   # for recording-level aggregation later

X_win     = np.asarray(X_win)
y_win     = np.asarray(y_win)
groups_win = np.asarray(groups_win)
rec_ids   = np.asarray(rec_ids)

print(f"Windowed dataset: X={X_win.shape}, "
      f"subjects={len(np.unique(groups_win))}, recordings={len(np.unique(rec_ids))}")
np.savez("features_sam40_windowed.npz",
         X=X_win, y=y_win, groups=groups_win, rec_ids=rec_ids)


Windowed dataset: X=(3960, 676), subjects=40, recordings=360


## GroupKFold Random Forest baseline

In [144]:
# ---- Subject-wise CV with recording-level aggregation -----------------------
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

gkf = GroupKFold(n_splits=5)
win_acc, rec_acc, rec_auc = [], [], []

for tr, te in gkf.split(X_win, y_win, groups_win):
    scaler = StandardScaler().fit(X_win[tr])
    clf = RandomForestClassifier(
        n_estimators=300, max_depth=None,
        class_weight="balanced", n_jobs=-1, random_state=42
    ).fit(scaler.transform(X_win[tr]), y_win[tr])

    proba = clf.predict_proba(scaler.transform(X_win[te]))[:, 1]
    pred  = (proba >= 0.5).astype(int)

    # window-level
    win_acc.append(accuracy_score(y_win[te], pred))

    # recording-level: average window probabilities per recording
    df = pd.DataFrame({"rec": rec_ids[te], "proba": proba, "y": y_win[te]})
    agg = df.groupby("rec").agg(proba=("proba", "mean"), y=("y", "first"))
    rec_pred = (agg["proba"] >= 0.5).astype(int)
    rec_acc.append(accuracy_score(agg["y"], rec_pred))
    rec_auc.append(roc_auc_score(agg["y"], agg["proba"]))

print(f"Window-level acc:    {np.mean(win_acc):.3f} ± {np.std(win_acc):.3f}")
print(f"Recording-level acc: {np.mean(rec_acc):.3f} ± {np.std(rec_acc):.3f}")
print(f"Recording-level AUC: {np.mean(rec_auc):.3f} ± {np.std(rec_auc):.3f}")


Window-level acc:    0.524 ± 0.043
Recording-level acc: 0.531 ± 0.055
Recording-level AUC: 0.535 ± 0.060


In [145]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score

def make_models():
    """Each model wrapped so scaling/imputation is fit inside CV folds only."""
    def pipe(clf):
        return Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale",  StandardScaler()),
            ("clf",    clf),
        ])
    return {
        "logreg_l2": pipe(LogisticRegression(max_iter=2000, C=1.0)),
        "logreg_l1": pipe(LogisticRegression(max_iter=2000, C=0.5,
                                             penalty="l1", solver="liblinear")),
        "lda":       pipe(LinearDiscriminantAnalysis()),
        "svm_rbf":   pipe(SVC(C=1.0, gamma="scale", probability=True)),
        "knn":       pipe(KNeighborsClassifier(n_neighbors=25)),
        # trees don't need scaling but harmless to keep the same pipe
        "rf":        RandomForestClassifier(n_estimators=400, max_depth=None,
                                            min_samples_leaf=5, n_jobs=-1,
                                            class_weight="balanced"),
        "gbdt":      GradientBoostingClassifier(n_estimators=300, max_depth=3,
                                                learning_rate=0.05),
    }

def evaluate(X, y, groups, rec_ids, n_splits=5, seed=0):
    """
    X: (n_windows, n_features)
    y: (n_windows,) window labels (== recording label)
    groups: (n_windows,) subject id  -> prevents subject leakage
    rec_ids: (n_windows,) recording id -> for recording-level aggregation
    """
    gkf = GroupKFold(n_splits=n_splits)
    results = {}

    for name, model in make_models().items():
        win_acc, rec_acc, rec_auc = [], [], []
        for tr, te in gkf.split(X, y, groups):
            model.fit(X[tr], y[tr])

            # window-level
            if hasattr(model, "predict_proba"):
                p = model.predict_proba(X[te])[:, 1]
            else:
                p = model.decision_function(X[te])
            yhat = (p >= 0.5).astype(int) if p.max() <= 1 else (p >= 0).astype(int)
            win_acc.append(accuracy_score(y[te], yhat))

            # recording-level: average window probs per recording
            r = rec_ids[te]
            rec_p, rec_y = [], []
            for rid in np.unique(r):
                m = r == rid
                rec_p.append(p[m].mean())
                rec_y.append(y[te][m][0])
            rec_p, rec_y = np.array(rec_p), np.array(rec_y)
            rec_acc.append(accuracy_score(rec_y, (rec_p >= np.median(rec_p)).astype(int)))
            if len(np.unique(rec_y)) > 1:
                rec_auc.append(roc_auc_score(rec_y, rec_p))

        results[name] = {
            "win_acc": (np.mean(win_acc), np.std(win_acc)),
            "rec_acc": (np.mean(rec_acc), np.std(rec_acc)),
            "rec_auc": (np.mean(rec_auc), np.std(rec_auc)) if rec_auc else (np.nan, np.nan),
        }

    # pretty print
    print(f"{'model':<12} {'win_acc':>14} {'rec_acc':>14} {'rec_auc':>14}")
    for name, r in results.items():
        print(f"{name:<12} "
              f"{r['win_acc'][0]:.3f}±{r['win_acc'][1]:.3f}  "
              f"{r['rec_acc'][0]:.3f}±{r['rec_acc'][1]:.3f}  "
              f"{r['rec_auc'][0]:.3f}±{r['rec_auc'][1]:.3f}")
    return results

# results = evaluate(X, y, groups, rec_ids)


In [146]:
make_models()

{'logreg_l2': Pipeline(steps=[('impute', SimpleImputer(strategy='median')),
                 ('scale', StandardScaler()),
                 ('clf', LogisticRegression(max_iter=2000))]),
 'logreg_l1': Pipeline(steps=[('impute', SimpleImputer(strategy='median')),
                 ('scale', StandardScaler()),
                 ('clf',
                  LogisticRegression(C=0.5, max_iter=2000, penalty='l1',
                                     solver='liblinear'))]),
 'lda': Pipeline(steps=[('impute', SimpleImputer(strategy='median')),
                 ('scale', StandardScaler()),
                 ('clf', LinearDiscriminantAnalysis())]),
 'svm_rbf': Pipeline(steps=[('impute', SimpleImputer(strategy='median')),
                 ('scale', StandardScaler()), ('clf', SVC(probability=True))]),
 'knn': Pipeline(steps=[('impute', SimpleImputer(strategy='median')),
                 ('scale', StandardScaler()),
                 ('clf', KNeighborsClassifier(n_neighbors=25))]),
 'rf': RandomForestClass